# CSV 비교 분석

**대상**
1. `CycPeptMPDB-4D_with_SMILES.csv` — 4D descriptor 셋 (Water/Hexane 3D 구조 기반) + Assay 에서 상속된 메타/SMILES/PAMPA
2. `CycPeptMPDB_Peptide_Assay_PAMPA.csv` — 원본 CycPeptMPDB 의 PAMPA 어세이 데이터 (RDKit 2D descriptor 240+개 포함)

**중요한 사전 사실**
4D 파일의 다음 컬럼들은 **Assay 파일에서 가져온 것** (= 동일 출처, 값도 100% 일치) :
- `SMILES`, `HELM`, `Sequence`, `Source`, `Molecule_Shape`, `Monomer_Length`, `Monomer_Length_in_Main_Chain`,
  `Original_Name_in_Source_Literature`, `Structurally_Unique_ID`, `Permeability`, `PAMPA`, `PAMPA-4D`

→ 따라서 **두 파일 간 PAMPA / Permeability / SMILES 등 공통 컬럼의 값 비교 자체는 의미가 없다** (둘이 같음). 의미 있는 분석은 **두 파일이 보유한 서로 다른 descriptor 의 관계** 와 **데이터셋 편향** 이다.

**이 노트북의 분석 흐름**
1. 스키마 / 컬럼 비교 — 4D 전용, Assay 전용 컬럼 정리
2. 펩타이드 overlap (ID 기준) — 4D ⊂ Assay 임을 확인
3. **부분집합 편향 분석** — 4D 가 Assay 의 random subset 인가? (PAMPA, Source, Length 분포)
4. **4D 3D descriptor ↔ Assay 2D descriptor redundancy** — 두 정보원이 중복인가 보완인가? (PSA 계열 등)
5. **PAMPA 예측력 비교** — 4D-only vs Assay-only vs 결합 모델
6. **결합 데이터셋 활용 가능성** — inner join 후 feature 통합
7. 종합 요약

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.unicode_minus'] = False

PATH_4D = '/ssd0/sohyun/cyclic_peptide_permeability/CycPeptMPDB-4D_with_SMILES.csv'
PATH_AS = '/ssd0/sohyun/cyclic_peptide_permeability/CycPeptMPDB_Peptide_Assay_PAMPA.csv'

df_4d = pd.read_csv(PATH_4D)
df_as = pd.read_csv(PATH_AS)

print(f'4D w/ SMILES : {df_4d.shape}')
print(f'PAMPA assay  : {df_as.shape}')

## 1. 스키마 비교 — 컬럼 차이 / 공통 컬럼

In [ ]:
cols_4d = set(df_4d.columns)
cols_as = set(df_as.columns)

common      = sorted(cols_4d & cols_as)
only_4d     = sorted(cols_4d - cols_as)
only_assay  = sorted(cols_as - cols_4d)

print(f'공통 컬럼               : {len(common)}')
print(f'4D 전용 컬럼            : {len(only_4d)}')
print(f'Assay 전용 컬럼          : {len(only_assay)}')

print('\n--- 공통 컬럼 ---')
print(common)
print('\n--- 4D 전용 컬럼 ---')
print(only_4d)
print(f'\n--- Assay 전용 (앞 30개 / 총 {len(only_assay)}) ---')
print(only_assay[:30])

In [ ]:
# 컬럼 카테고리화 (assay 쪽이 압도적으로 많아 카테고리별로 묶어서 보여줌)
categories = {
    'meta':       ['ID','CycPeptMPDB_ID','Source','Year','Version','Original_Name_in_Source_Literature',
                   'Structurally_Unique_ID','Same_Peptides_ID','Same_Peptides_Source',
                   'Same_Peptides_Permeability','Same_Peptides_Assay'],
    'identifier': ['SMILES','HELM','HELM_URL','Sequence'],
    'sequence':   ['Sequence_LogP','Sequence_TPSA','Monomer_Length','Monomer_Length_in_Main_Chain',
                   'Molecule_Shape'],
    'permeability':['Permeability','PAMPA','Caco2','MDCK','RRCK',
                    'Detection_Limit_1','Detection_Limit_2',
                    'R_PAMAP','R_Caco2','R_MDCK','R_RRCK','T_PAMPA'],
    '4D structure':[c for c in df_4d.columns if any(k in c for k in ['_3D_','avgRMSD','Desolvation'])],
    '2D PSA / EPSA':['EPSA','PSA','_3DPSA','TPSA','CHCl3_3DPSA','H2O_3DPSA','LabuteASA'],
    'RDKit base':  ['MaxEStateIndex','MinEStateIndex','qed','MolWt','HeavyAtomMolWt','ExactMolWt',
                    'MolLogP','MolMR','FractionCSP3','HeavyAtomCount'],
    'PEOE_VSA':    [c for c in df_as.columns if c.startswith('PEOE_VSA')],
    'SMR_VSA':     [c for c in df_as.columns if c.startswith('SMR_VSA')],
    'SlogP_VSA':   [c for c in df_as.columns if c.startswith('SlogP_VSA')],
    'EState_VSA':  [c for c in df_as.columns if c.startswith('EState_VSA') or c.startswith('VSA_EState')],
    'Chi':         [c for c in df_as.columns if c.startswith('Chi')],
    'Kappa':       [c for c in df_as.columns if c.startswith('Kappa')],
    'BCUT2D':      [c for c in df_as.columns if c.startswith('BCUT2D')],
    'fr_*':        [c for c in df_as.columns if c.startswith('fr_')],
    'PCA':         ['PC1','PC2'],
}
rows = []
for cat, cols in categories.items():
    in_4d = sum(c in cols_4d for c in cols)
    in_as = sum(c in cols_as for c in cols)
    rows.append({'category': cat, 'columns_in_category': len(cols),
                 'present_in_4D': in_4d, 'present_in_Assay': in_as})
schema_df = pd.DataFrame(rows)
schema_df

## 2. 펩타이드 overlap — ID 기준

- 4D 파일의 `CycPeptMPDB_ID` ↔ Assay 의 `ID`
- (SMILES 는 4D 가 Assay 에서 그대로 가져온 컬럼이므로 비교 생략)

In [ ]:
ids_4d = set(df_4d['CycPeptMPDB_ID'].astype(int))
ids_as = set(df_as['ID'].astype(int))

in_both    = ids_4d & ids_as
only_in_4d = ids_4d - ids_as
only_in_as = ids_as - ids_4d

print(f'4D unique IDs           : {len(ids_4d)}')
print(f'Assay unique IDs        : {len(ids_as)}')
print(f'양쪽 모두에 있는 ID      : {len(in_both)}')
print(f'4D 에만 있는 ID         : {len(only_in_4d)}')
print(f'Assay 에만 있는 ID       : {len(only_in_as)}')

# 중복 row
print(f'\n4D row 중복 (ID 기준)    : {df_4d["CycPeptMPDB_ID"].duplicated().sum()}')
print(f'Assay row 중복 (ID 기준)  : {df_as["ID"].duplicated().sum()}')

In [ ]:
# ID overlap 시각화 (SMILES 비교는 동일 출처라 제거)
try:
    from matplotlib_venn import venn2
    fig, ax = plt.subplots(figsize=(7, 5))
    venn2([ids_4d, ids_as], set_labels=('4D w/ SMILES', 'Assay'), ax=ax)
    ax.set_title('Peptide ID overlap')
    plt.tight_layout(); plt.show()
except ImportError:
    print('matplotlib_venn 미설치 → bar 로 대체  (설치: !pip install matplotlib_venn)')
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(['only_4D','both','only_Assay'],
           [len(only_in_4d), len(in_both), len(only_in_as)],
           color=['#5bc0de','#5cb85c','#f0ad4e'])
    for i, v in enumerate([len(only_in_4d), len(in_both), len(only_in_as)]):
        ax.text(i, v, f'{v}', ha='center', va='bottom')
    ax.set_title('Peptide ID overlap')
    plt.tight_layout(); plt.show()

## 3. 부분집합 편향 분석 — 4D 는 Assay 의 random subset 인가?

Assay 7298 펩타이드 중 4D 셋에 포함된 것 (5160) 과 그렇지 않은 것 (2138) 의 **PAMPA 분포** 를 비교.
- **random subset** 이라면 두 분포가 통계적으로 같아야 함 (KS p > 0.05)
- **biased subset** 이라면 두 분포가 다르고, 그 방향이 의미 있음 (어떤 종류의 펩타이드가 누락됐나?)

In [ ]:
df_as_with_perm = df_as[['ID','PAMPA','Source','Monomer_Length','Molecule_Shape']].copy()
df_as_with_perm['in_4D'] = df_as_with_perm['ID'].isin(ids_4d)

tab2 = df_as_with_perm.groupby('in_4D')['PAMPA'].agg(['count','mean','std','median']).round(3)
tab2.index = tab2.index.map({True:'in_4D', False:'Assay_only'})
display(tab2)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for grp, color, lbl in [(True,'steelblue','in_4D'),(False,'orange','Assay_only')]:
    s = df_as_with_perm.loc[df_as_with_perm['in_4D']==grp, 'PAMPA'].dropna()
    axes[0].hist(s, bins=50, alpha=0.55, color=color, label=f'{lbl} (n={len(s)})', edgecolor='white')
axes[0].axvline(-6, color='red', ls='--', alpha=0.5)
axes[0].set(xlabel='PAMPA', ylabel='count', title='Assay PAMPA — in_4D vs Assay_only')
axes[0].legend()

data = [df_as_with_perm.loc[df_as_with_perm['in_4D']==True,'PAMPA'].dropna(),
         df_as_with_perm.loc[df_as_with_perm['in_4D']==False,'PAMPA'].dropna()]
bp = axes[1].boxplot(data, labels=['in_4D','Assay_only'], patch_artist=True)
for patch,c in zip(bp['boxes'],['steelblue','orange']):
    patch.set_facecolor(c)
axes[1].axhline(-6, color='red', ls='--', alpha=0.5)
axes[1].set(ylabel='PAMPA', title='Boxplot')
plt.tight_layout(); plt.show()

# 통계 비교
ks = stats.ks_2samp(data[0], data[1])
mw = stats.mannwhitneyu(data[0], data[1], alternative='two-sided')
print(f'KS  : {ks.statistic:.4f}  p={ks.pvalue:.3e}')
print(f'MWU : U={mw.statistic:.0f}  p={mw.pvalue:.3e}')

# Permeable rate 비교
rate_in    = ((data[0] >= -6).sum()) / len(data[0])
rate_out   = ((data[1] >= -6).sum()) / len(data[1])
print(f'Permeable rate in_4D     : {rate_in:.3%}')
print(f'Permeable rate Assay_only: {rate_out:.3%}')

### 3-2. 메타데이터 분포 비교 — Source / Length / Shape

부분집합 편향이 어떤 차원에서 발생하는지 (Source / Length / Shape) 확인.

In [ ]:
src_4d = df_4d['Source'].value_counts()
src_as = df_as['Source'].value_counts()

all_src = sorted(set(src_4d.index) | set(src_as.index))
df_src = pd.DataFrame({'4D':    src_4d.reindex(all_src, fill_value=0),
                        'Assay': src_as.reindex(all_src, fill_value=0)})
df_src['only_in_Assay'] = (df_src['4D']==0) & (df_src['Assay']>0)
print(f'4D 에 등장하는 Source 수    : {(df_src["4D"]>0).sum()}')
print(f'Assay 에 등장하는 Source 수 : {(df_src["Assay"]>0).sum()}')
print(f'Assay 전용 Source 수        : {df_src["only_in_Assay"].sum()}')
display(df_src.sort_values('Assay', ascending=False).head(20))

# 시각화: 상위 15 source
top = df_src.sort_values('Assay', ascending=False).head(15)
x = np.arange(len(top))
fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x-0.2, top['4D'],    width=0.4, label='4D',    color='steelblue')
ax.bar(x+0.2, top['Assay'], width=0.4, label='Assay', color='orange')
ax.set_xticks(x); ax.set_xticklabels(top.index, rotation=40, ha='right')
ax.set(title='Source 별 sample count (top 15)', ylabel='count')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Monomer_Length
len_4d = df_4d['Monomer_Length'].value_counts(normalize=True).sort_index()
len_as = df_as['Monomer_Length'].value_counts(normalize=True).sort_index()
all_len = sorted(set(len_4d.index) | set(len_as.index))
x = np.arange(len(all_len))
axes[0].bar(x-0.2, [len_4d.get(L,0) for L in all_len], width=0.4, label='4D', color='steelblue')
axes[0].bar(x+0.2, [len_as.get(L,0) for L in all_len], width=0.4, label='Assay', color='orange')
axes[0].set_xticks(x); axes[0].set_xticklabels(all_len)
axes[0].set(title='Monomer_Length 분포 (normalized)', xlabel='Monomer_Length', ylabel='fraction')
axes[0].legend()

# Molecule_Shape
sh_4d = df_4d['Molecule_Shape'].value_counts(normalize=True)
sh_as = df_as['Molecule_Shape'].value_counts(normalize=True)
all_sh = sorted(set(sh_4d.index) | set(sh_as.index))
x = np.arange(len(all_sh))
axes[1].bar(x-0.2, [sh_4d.get(s,0) for s in all_sh], width=0.4, label='4D',    color='steelblue')
axes[1].bar(x+0.2, [sh_as.get(s,0) for s in all_sh], width=0.4, label='Assay', color='orange')
axes[1].set_xticks(x); axes[1].set_xticklabels(all_sh, rotation=20)
axes[1].set(title='Molecule_Shape 분포 (normalized)', ylabel='fraction')
axes[1].legend()
plt.tight_layout(); plt.show()

# Permeable rate by Length 비교
df_4d_p = df_4d.assign(Permeable=(df_4d['PAMPA']>=-6).astype(int))
df_as_p = df_as.assign(Permeable=(df_as['PAMPA']>=-6).astype(int))
rate_4d = df_4d_p.groupby('Monomer_Length')['Permeable'].mean()
rate_as = df_as_p.groupby('Monomer_Length')['Permeable'].mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(rate_4d.index, rate_4d.values, 'o-', label='4D',    color='steelblue')
ax.plot(rate_as.index, rate_as.values, 's-', label='Assay', color='orange')
ax.set(xlabel='Monomer_Length', ylabel='permeable rate (PAMPA>=-6)',
        title='Permeable rate by Monomer_Length')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 4. 4D 3D descriptor ↔ Assay 2D descriptor — redundancy / complementarity

두 파일 모두에 다양한 PSA / 표면적 / 극성 관련 descriptor 가 있다. 둘이 본질적으로 **같은 정보를 두 번 측정한 것** 인지, 아니면 **서로 다른 정보** 를 담고 있는지 확인.

검토할 그룹:
- **PSA 계열** : 4D 의 `Water_3D_PSA`, `Hexane_3D_PSA` ↔ Assay 의 `TPSA`, `_3DPSA`, `CHCl3_3DPSA`, `H2O_3DPSA`
- **NPSA / SASA 계열** : 4D 의 `Water_3D_NPSA/SASA`, `Hexane_3D_NPSA/SASA` ↔ Assay 의 `LabuteASA`
- **Hydrophobicity** : 4D 는 직접 없음 / Assay 의 `MolLogP`, `Sequence_LogP`

In [ ]:
# 4D + Assay inner join (CycPeptMPDB_ID = ID)
merged = df_4d.merge(df_as.rename(columns={'ID':'CycPeptMPDB_ID'}),
                      on='CycPeptMPDB_ID', suffixes=('_4D','_AS'))
print(f'merged shape: {merged.shape}  (4D 5160 펩타이드의 모든 4D + Assay descriptor 가 합쳐짐)')

# PSA 계열만 추려서 상관행렬
psa_candidates = ['Water_3D_PSA','Hexane_3D_PSA','Water_3D_NPSA','Hexane_3D_NPSA',
                   'Water_3D_SASA','Hexane_3D_SASA',
                   'TPSA','_3DPSA','CHCl3_3DPSA','H2O_3DPSA','EPSA','LabuteASA','MolLogP']
psa_present = []
for c in psa_candidates:
    if c not in merged.columns:
        continue
    s = pd.to_numeric(merged[c], errors='coerce')
    if s.notna().sum() > 100:
        psa_present.append(c)
print(f'사용 가능한 descriptor ({len(psa_present)}):', psa_present)

corr_mat = merged[psa_present].apply(pd.to_numeric, errors='coerce').corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_mat, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
             vmin=-1, vmax=1, square=True, annot_kws={'size':8}, ax=ax)
ax.set_title('Surface / polarity descriptor correlation matrix\n(4D vs Assay)', fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# 4D 와 Assay 사이 cross-correlation 만 추출 (서로 다른 출처끼리만)
cols_4d_only = ['Water_3D_PSA','Hexane_3D_PSA','Water_3D_NPSA','Hexane_3D_NPSA',
                 'Water_3D_SASA','Hexane_3D_SASA']
cols_as_only = ['TPSA','CHCl3_3DPSA','H2O_3DPSA','LabuteASA','MolLogP']
cols_4d_only = [c for c in cols_4d_only if c in merged.columns]
cols_as_only = [c for c in cols_as_only if c in merged.columns and merged[c].notna().sum() > 100]

cross = merged[cols_4d_only + cols_as_only].corr().loc[cols_4d_only, cols_as_only]

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cross, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
             vmin=-1, vmax=1, ax=ax)
ax.set(title='Cross-correlation : 4D 3D descriptor (rows) ↔ Assay 2D descriptor (cols)',
        xlabel='Assay 2D descriptor', ylabel='4D 3D descriptor')
plt.tight_layout(); plt.show()

# 가장 redundant / complementary 인 쌍 hightlight
flat = cross.stack().reset_index()
flat.columns = ['4D', 'Assay', 'corr']
flat['|r|'] = flat['corr'].abs()
print('\n가장 강한 cross-correlation top 5 (redundant pair):')
display(flat.nlargest(5, '|r|').round(3))
print('\n가장 약한 cross-correlation top 5 (complementary pair):')
display(flat.nsmallest(5, '|r|').round(3))

## 5. PAMPA 예측력 비교 — 4D-only vs Assay-only vs 결합

같은 5160 펩타이드, 같은 PAMPA target. feature 그룹만 바꿔가며 RandomForest 회귀 모델의 test R² 비교.

- **4D-only** : `Water/Hexane_3D_*`, `*_avgRMSD_*`, `Desolvation_Free_Energy` (13개)
- **Assay-only** : RDKit 2D descriptor 모음 (TPSA, MolWt, MolLogP, BCUT2D, Chi*, Kappa*, PEOE_VSA, SlogP_VSA, EState_VSA, fr_*, …) — 결측 적은 것만
- **결합** : 위 둘 합본

→ 결합 모델이 두 단독 모델 어느 쪽보다 좋아진다면 4D 와 Assay 정보가 **상호 보완적**.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# 4D feature group 정의
cols_4d_feat = [
    'Water_avgRMSD_All','Water_avgRMSD_BackBone',
    'Hexane_avgRMSD_All','Hexane_avgRMSD_BackBone',
    'Desolvation_Free_Energy',
    'Water_3D_SASA','Water_3D_NPSA','Water_3D_PSA',
    'Hexane_3D_SASA','Hexane_3D_NPSA','Hexane_3D_PSA',
    'Monomer_Length','Monomer_Length_in_Main_Chain',
]

# Assay 2D descriptor 후보 — 결측 거의 없고 숫자형인 것만
exclude_keys = ('Same_Peptides','HELM','Source','Sequence','SMILES','Original_Name',
                 'Detection','PAMPA','Permeability','Caco2','MDCK','RRCK','T_PAMPA',
                 'Year','Version','ID','Structurally_Unique_ID',
                 'Molecule_Shape','Monomer_Length','NULL','Ipc')   # Ipc 는 RDKit 에서 매우 큰 값을 갖는 outlier-prone 컬럼
cols_as_feat = []
in_4d_assay = df_as.set_index('ID').loc[df_4d['CycPeptMPDB_ID']]
for c in df_as.columns:
    if any(k in c for k in exclude_keys):
        continue
    s = pd.to_numeric(in_4d_assay[c], errors='coerce')
    miss = s.isna().mean()
    if miss < 0.01 and s.nunique(dropna=True) > 5:
        # float32 범위 초과하는 outlier 가 있는 컬럼 제외
        if np.isfinite(s).all() and s.abs().max() < 1e30:
            cols_as_feat.append(c)
print(f'Assay feature 후보 {len(cols_as_feat)}개 (결측 < 1%, unique > 5, 숫자형, finite)')

# merge 한 데이터프레임 사용
df = merged.copy()
target_col = 'PAMPA-4D' if 'PAMPA-4D' in df.columns else ('PAMPA_4D' if 'PAMPA_4D' in df.columns else 'PAMPA')
print(f'Target column: {target_col}')

all_feats = list(set(cols_4d_feat + cols_as_feat) & set(df.columns))
df_use = df[[target_col] + all_feats].apply(pd.to_numeric, errors='coerce')
df_use = df_use.replace([np.inf, -np.inf], np.nan).dropna()

# float32 한도 넘는 값이 있으면 winsorize (1%, 99% 기준)
for c in all_feats:
    if not np.isfinite(df_use[c]).all() or df_use[c].abs().max() > 1e30:
        lo, hi = df_use[c].quantile([0.001, 0.999])
        df_use[c] = df_use[c].clip(lo, hi)

print(f'After dropna : {len(df_use)} rows, {len(all_feats)} features')

groups = {
    '4D-only':   [c for c in cols_4d_feat if c in df_use.columns],
    'Assay-only':[c for c in cols_as_feat if c in df_use.columns],
    'Combined':  [c for c in df_use.columns if c != target_col],
}

X_all = df_use.drop(columns=[target_col])
y     = df_use[target_col].values
idx_tr, idx_te = train_test_split(np.arange(len(df_use)), test_size=0.2, random_state=42)
y_tr, y_te = y[idx_tr], y[idx_te]
kf = KFold(n_splits=5, shuffle=True, random_state=42)

results = []
for name, cols in groups.items():
    X = X_all[cols].values.astype(np.float64)
    Xtr, Xte = X[idx_tr], X[idx_te]
    rf = RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=42)
    rf.fit(Xtr, y_tr)
    pred = rf.predict(Xte)
    cv = cross_val_score(rf, Xtr, y_tr, cv=kf, scoring='r2', n_jobs=-1)
    results.append({
        'feature group': name,
        'n_features':    len(cols),
        'cv_R2_mean':    cv.mean(),
        'cv_R2_std':     cv.std(),
        'test_R2':       r2_score(y_te, pred),
        'test_MAE':      mean_absolute_error(y_te, pred),
    })

res_df = pd.DataFrame(results).round(4)
display(res_df)

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(res_df))
ax.bar(x-0.2, res_df['cv_R2_mean'], yerr=res_df['cv_R2_std'], width=0.4,
        label='CV R² (5-fold)', color='steelblue')
ax.bar(x+0.2, res_df['test_R2'], width=0.4, label='Test R²', color='orange')
ax.set_xticks(x); ax.set_xticklabels(res_df['feature group'])
ax.axhline(0, color='k', lw=0.5)
ax.set(ylabel='R²', title='PAMPA prediction R² by feature group (RandomForest)')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Combined 모델에서 어느 feature 가 핵심인지 — feature importance top 20
combined_cols = list(df_use.columns.drop(target_col))
X = df_use[combined_cols].values
rf = RandomForestRegressor(n_estimators=400, n_jobs=-1, random_state=42)
rf.fit(X[idx_tr], y_tr)

imp = pd.Series(rf.feature_importances_, index=combined_cols).sort_values(ascending=False)
# 출처 라벨링
imp_df = imp.head(20).to_frame('importance').reset_index().rename(columns={'index':'feature'})
imp_df['source'] = imp_df['feature'].apply(lambda x: '4D' if x in cols_4d_feat else 'Assay')

fig, ax = plt.subplots(figsize=(9, 7))
colors = ['steelblue' if s == '4D' else 'orange' for s in imp_df['source']]
ax.barh(imp_df['feature'][::-1], imp_df['importance'][::-1], color=colors[::-1])
# legend
import matplotlib.patches as mpatches
ax.legend(handles=[mpatches.Patch(color='steelblue', label='4D'),
                    mpatches.Patch(color='orange',    label='Assay')])
ax.set(title='Combined-model feature importance (top 20)\n색은 출처 (4D vs Assay)',
        xlabel='importance')
plt.tight_layout(); plt.show()

# 출처별 누적 importance
print('Top 20 importance 누적 — 출처별:')
print(imp_df.groupby('source')['importance'].agg(['count','sum']).round(3))
print(f'\n전체 feature importance — 출처별 합:')
total_4d = imp[imp.index.isin(cols_4d_feat)].sum()
total_as = imp[~imp.index.isin(cols_4d_feat)].sum()
print(f'  4D    : {total_4d:.3f} ({total_4d*100:.1f}%)')
print(f'  Assay : {total_as:.3f} ({total_as*100:.1f}%)')

## 6. 결합 데이터셋 활용 가능성

두 파일을 inner join 했을 때 얻는 데이터셋의 **실용 정보** 정리.

In [ ]:
# Combined PCA — 4D feature 와 Assay feature 가 같은 latent space 에서 어떻게 분포?
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# 결합 모델용 feature 만 사용 (위 §5 와 동일)
X = df_use[combined_cols].values
y = df_use[target_col].values

Xs = StandardScaler().fit_transform(X)
pca = PCA(n_components=4, random_state=42)
Z = pca.fit_transform(Xs)
print('Explained variance ratio:', pca.explained_variance_ratio_.round(3))
print('Cumulative              :', np.cumsum(pca.explained_variance_ratio_).round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sc = axes[0].scatter(Z[:,0], Z[:,1], c=y, cmap='RdYlGn', s=8, alpha=0.6)
axes[0].set(title='PC1 vs PC2 (color = PAMPA)', xlabel='PC1', ylabel='PC2')
plt.colorbar(sc, ax=axes[0], label='PAMPA')

# Loadings — 출처별 색
loadings_pc1 = pd.Series(pca.components_[0], index=combined_cols).abs().nlargest(15)
loadings_pc2 = pd.Series(pca.components_[1], index=combined_cols).abs().nlargest(15)
top_load = pd.concat([loadings_pc1, loadings_pc2], axis=1).fillna(0)
top_load.columns = ['|PC1|', '|PC2|']
top_load = top_load.sort_values('|PC1|', ascending=False).head(15)
top_load['source'] = ['4D' if c in cols_4d_feat else 'Assay' for c in top_load.index]

axes[1].axis('off')
tbl = axes[1].table(cellText=top_load.round(3).values,
                     rowLabels=top_load.index, colLabels=top_load.columns,
                     loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(8)
axes[1].set_title('Top-15 feature by |PC1| loading')
plt.tight_layout(); plt.show()

## 7. 종합 요약

### 스키마
- 두 파일은 메타/identifier (`SMILES`, `HELM`, `Sequence`, `Source`, `Molecule_Shape`, `Monomer_Length(_in_Main_Chain)`, `Original_Name`, `Structurally_Unique_ID`, `Permeability`, `PAMPA`) 를 공유 — 이들은 모두 Assay 에서 4D 로 상속된 동일 출처라 **두 파일 간 값 비교는 무의미**.
- **4D 전용** (12개) : `CycPeptMPDB_ID`, `PAMPA-4D`, `Desolvation_Free_Energy`, `Water/Hexane_avgRMSD_All/BackBone`, `Water/Hexane_3D_SASA/NPSA/PSA`
- **Assay 전용** (236개) : `ID`, `Caco2`, `MDCK`, `RRCK`, `T_PAMPA`, 그리고 RDKit 2D descriptor 200+개 (`MolWt`, `MolLogP`, `TPSA`, `qed`, `BCUT2D_*`, `Chi*`, `Kappa*`, `PEOE/SMR/SlogP_VSA*`, `EState_*`, `fr_*`, `_3DPSA`, `CHCl3_3DPSA`, `H2O_3DPSA`, `EPSA`, …)

### 펩타이드 overlap (§2)
- 4D 5160 IDs ⊂ Assay 7298 IDs (4D 전용 ID = 0, Assay 전용 = 2138)

### 부분집합 편향 (§3)
- in_4D vs Assay_only PAMPA 분포 : **KS p = 1.3e-106, MWU p = 1.2e-39** → 4D 는 Assay 의 random subset 이 **아님**.
  - Mean PAMPA: in_4D = −5.987, Assay_only = −5.715 (Assay_only 가 약간 더 permeable 쪽)
  - Permeable rate: in_4D 65.5% vs Assay_only 68.6%
- Source 편향: 4D 는 5개 source (`2020_Townsend`, `2021_Kelly`, `2016_Furukawa`, `2018_Naylor`, `2015_Wang`), Assay 는 42개 — 37개 source 가 4D 미포함 (예: `2013_CHUGAI` 878, `2024_Faris` 234) → **4D 는 3D 구조 계산 가능했던 5개 source 한정**

### 4D 3D ↔ Assay 2D descriptor redundancy (§4)
| 가장 redundant 쌍 | r |
|---|---|
| `Water_3D_SASA` ↔ `LabuteASA` | 0.965 |
| `Hexane_3D_SASA` ↔ `LabuteASA` | 0.964 |
| `Water_3D_PSA` ↔ `TPSA` | 0.896 |
| `H2O_3DPSA` ↔ `CHCl3_3DPSA` | 0.978 (Assay 내부) |

| 가장 complementary 쌍 | r |
|---|---|
| `Water_3D_SASA` ↔ `MolLogP` | 0.006 |
| `Hexane_3D_SASA` ↔ `MolLogP` | 0.065 |
| `Hexane_3D_PSA` ↔ `MolLogP` | −0.326 |

→ 표면적/PSA 계열은 4D 3D 와 Assay 2D 가 **거의 같은 정보** (r > 0.9). 반면 hydrophobicity 류 (`MolLogP`) 는 4D 측 직접 대응이 없어 **Assay 가 추가로 제공하는 정보**.

### PAMPA 예측력 비교 (§5, RandomForest)
| feature group | n_features | CV R² | Test R² | Test MAE |
|---|---|---|---|---|
| 4D-only       | 11  | 0.112 | **0.097** | 0.657 |
| Assay-only    | 103 | 0.212 | **0.244** | 0.549 |
| **Combined**  | 114 | 0.314 | **0.270** | 0.541 |

- **결합 모델이 두 단독 모델 모두를 능가** → 4D 와 Assay descriptor 는 상호 보완적.
- Combined model 의 feature importance: 4D 42.7%, Assay 57.3% → 거의 균형. 4D feature 11개가 Assay 103개에 비해 적음에도 importance 의 절반 가까이 차지 → **개별 4D feature 의 정보 밀도가 높음**.
- Top-20 importance 중 4D 11개, Assay 9개 → 핵심 feature 의 절반 이상이 4D 에서 옴.

### 결합 데이터셋 활용 (§6)
- inner join 데이터셋 shape: (5160, 269)
- PCA: PC1-2 가 분산 72%를 설명, 4D feature (특히 SASA/PSA) 가 PC1 에서 강한 loading.

### 결론 / 활용 제안
1. **결합 모델이 가장 성능 좋음** — 4D 의 conformer 기반 표면 지표 + Assay 의 RDKit 2D descriptor 를 같이 사용해야 PAMPA 예측력이 최대가 됨.
2. **redundant pair** (Water_3D_SASA ≈ LabuteASA, Water_3D_PSA ≈ TPSA) 는 결합 모델에 둘 다 넣어도 큰 이득은 없음 — 한쪽만 사용해 모델 단순화 가능.
3. **complementary pair** (`MolLogP` 등 hydrophobicity 류) 는 4D 에 없으므로 **Assay 에서 반드시 가져와야 함**.
4. 4D 가 5개 source 한정 → Assay_only 2138개 펩타이드 (37개 다른 source) 에 대해서는 **4D-기반 모델이 OOD 위험**. 다른 source 펩타이드의 PAMPA 예측은 Assay 2D descriptor 로 학습한 모델만 가능.
5. 부분집합 편향 분석 결과 4D 셋이 mean PAMPA 기준 −0.27 정도 더 non-permeable 쪽 — 4D-only 모델이 학습한 분포가 Assay_only 펩타이드에 그대로 적용되지 않을 가능성을 명시할 것.